In [5]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler

# ---------------------------------------------------
# PAGE CONFIG
# ---------------------------------------------------
st.set_page_config(
    page_title="South Korea Top 50 Analytics",
    page_icon="🎵",
    layout="wide"
)

st.title("🎵 South Korea Top 50 Playlist Analytics")
st.markdown("### Comeback Momentum, Chart Re-Entry & Fandom Intensity Analysis")

# ---------------------------------------------------
# LOAD DATA
# ---------------------------------------------------
uploaded_file = st.sidebar.file_uploader(
    "Upload CSV Dataset",
    type=["csv"]
)

if uploaded_file is None:
    st.info("Upload the South Korea Top 50 Playlist CSV file.")
    st.stop()

df = pd.read_csv('Atlantic_South_Korea.csv')

# ---------------------------------------------------
# DATA PREPROCESSING
# ---------------------------------------------------
df["date"] = pd.to_datetime(
    df["date"],
    dayfirst=True
)

df["song_artist"] = (
    df["song"].astype(str)
    + " - "
    + df["artist"].astype(str)
)

df["duration_min"] = df["duration_ms"] / 60000

# ---------------------------------------------------
# SIDEBAR FILTERS
# ---------------------------------------------------
st.sidebar.header("Filters")

artists = sorted(df["artist"].dropna().unique())

selected_artist = st.sidebar.selectbox(
    "Artist",
    ["All"] + artists
)

album_filter = st.sidebar.multiselect(
    "Album Type",
    options=df["album_type"].dropna().unique(),
    default=list(df["album_type"].dropna().unique())
)

date_range = st.sidebar.date_input(
    "Date Range",
    [df["date"].min(), df["date"].max()]
)

filtered = df.copy()

if selected_artist != "All":
    filtered = filtered[
        filtered["artist"] == selected_artist
    ]

filtered = filtered[
    filtered["album_type"].isin(album_filter)
]

filtered = filtered[
    (filtered["date"] >= pd.to_datetime(date_range[0]))
    &
    (filtered["date"] <= pd.to_datetime(date_range[1]))
]

# ---------------------------------------------------
# KPI CALCULATIONS
# ---------------------------------------------------
song_dates = (
    filtered
    .sort_values(["song_artist", "date"])
)

song_dates["gap"] = (
    song_dates
    .groupby("song_artist")["date"]
    .diff()
    .dt.days
)

song_dates["reentry"] = np.where(
    song_dates["gap"] > 1,
    1,
    0
)

reentry_df = (
    song_dates
    .groupby("song_artist")["reentry"]
    .sum()
    .reset_index()
)

rank_stats = (
    filtered
    .groupby("song_artist")["position"]
    .agg(["min", "max", "mean"])
    .reset_index()
)

rank_stats["rank_jump"] = (
    rank_stats["max"]
    - rank_stats["min"]
)

pop_stats = (
    filtered
    .groupby("song_artist")
    .agg(
        avg_popularity=("popularity", "mean"),
        max_popularity=("popularity", "max")
    )
    .reset_index()
)

momentum = pd.merge(
    rank_stats,
    pop_stats,
    on="song_artist"
)

momentum["momentum_score"] = (
    momentum["rank_jump"]
    * momentum["mean"]
)

# ---------------------------------------------------
# FANDOM INTENSITY SCORE
# ---------------------------------------------------
fandom = pd.merge(
    reentry_df,
    momentum[
        [
            "song_artist",
            "momentum_score"
        ]
    ],
    on="song_artist"
)

scaler = MinMaxScaler()

fandom[
    ["reentry_scaled",
     "momentum_scaled"]
] = scaler.fit_transform(
    fandom[
        ["reentry",
         "momentum_score"]
    ]
)

fandom["fandom_score"] = (
    fandom["reentry_scaled"] * 0.6
    +
    fandom["momentum_scaled"] * 0.4
)

# ---------------------------------------------------
# KPI CARDS
# ---------------------------------------------------
col1, col2, col3, col4 = st.columns(4)

col1.metric(
    "Songs",
    filtered["song_artist"].nunique()
)

col2.metric(
    "Artists",
    filtered["artist"].nunique()
)

col3.metric(
    "Total Records",
    len(filtered)
)

col4.metric(
    "Re-Entries",
    int(reentry_df["reentry"].sum())
)

st.divider()

# ---------------------------------------------------
# RE-ENTRY ANALYSIS
# ---------------------------------------------------
st.header("📈 Re-Entry Analysis")

top_reentry = (
    reentry_df
    .sort_values(
        "reentry",
        ascending=False
    )
    .head(15)
)

fig1 = px.bar(
    top_reentry,
    x="song_artist",
    y="reentry",
    title="Top Re-Entered Songs"
)

st.plotly_chart(
    fig1,
    use_container_width=True
)

# ---------------------------------------------------
# MOMENTUM ANALYSIS
# ---------------------------------------------------
st.header("🚀 Momentum Analysis")

top_momentum = (
    momentum
    .sort_values(
        "momentum_score",
        ascending=False
    )
    .head(15)
)

fig2 = px.bar(
    top_momentum,
    x="song_artist",
    y="momentum_score",
    title="Momentum Spike Score"
)

st.plotly_chart(
    fig2,
    use_container_width=True
)

# ---------------------------------------------------
# POPULARITY TREND
# ---------------------------------------------------
st.header("🔥 Popularity Trends")

daily_popularity = (
    filtered
    .groupby("date")["popularity"]
    .mean()
    .reset_index()
)

fig3 = px.line(
    daily_popularity,
    x="date",
    y="popularity",
    title="Average Popularity Over Time"
)

st.plotly_chart(
    fig3,
    use_container_width=True
)

# ---------------------------------------------------
# CONTENT ANALYSIS
# ---------------------------------------------------
st.header("🎧 Content Analysis")

album_chart = (
    filtered
    .groupby("album_type")
    .agg(
        avg_popularity=("popularity", "mean")
    )
    .reset_index()
)

fig4 = px.pie(
    album_chart,
    names="album_type",
    values="avg_popularity",
    title="Album Type Distribution"
)

st.plotly_chart(
    fig4,
    use_container_width=True
)

# Explicit Analysis
explicit_chart = (
    filtered
    .groupby("is_explicit")
    .agg(
        popularity=("popularity", "mean")
    )
    .reset_index()
)

fig5 = px.bar(
    explicit_chart,
    x="is_explicit",
    y="popularity",
    title="Explicit vs Non-Explicit Popularity"
)

st.plotly_chart(
    fig5,
    use_container_width=True
)

# ---------------------------------------------------
# FANDOM LEADERBOARD
# ---------------------------------------------------
st.header("🏆 Fandom Intensity Leaderboard")

leaderboard = (
    fandom
    .sort_values(
        "fandom_score",
        ascending=False
    )
)

st.dataframe(
    leaderboard.head(20),
    use_container_width=True
)

# ---------------------------------------------------
# DOWNLOAD SECTION
# ---------------------------------------------------
st.header("📥 Download Reports")

csv = leaderboard.to_csv(index=False)

st.download_button(
    label="Download Fandom Leaderboard CSV",
    data=csv,
    file_name="fandom_leaderboard.csv",
    mime="text/csv"
)

st.success("Dashboard Loaded Successfully")

2026-06-14 14:35:42.069 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 14:35:42.078 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 14:35:42.085 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 14:35:42.097 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 14:35:42.101 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 14:35:42.109 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 14:35:42.115 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-14 14:35:42.119 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()